# Setup — install what the pipeline needs

Run this notebook **once**, before Part 1 and Part 2, from the repository's `0_setup/` folder. It checks and can install the Python packages, then downloads the model grids into your workspace, `<DOWNLOADS>/WDMS_fitting_pipeline/`. At the end it runs the pipeline's own input check.

| What | Where | Size | How |
|---|---|---|---|
| Model tables (Bédard, BHAC15, La Plata) and SPECULOOS filters | `2_sed_dilution/tables/`, `2_sed_dilution/filters/` | 3 MB | **already in the repository**, nothing to do |
| Koester DA white-dwarf spectra, 1 066 files | `<WORKSPACE>/models/koester2/` | ~230 MB downloaded, ~140 MB on disk | **required**. Downloaded here from the SVO Theoretical Spectra Server (model `koester2`) |
| STScI PHOENIX grid (M dwarf) | `<WORKSPACE>/model_cache/` | ~390 MB | needed with the default settings (`SED_MS_ATMOSPHERE = "phoenix"`). Downloaded here, or automatically at the first run |
| BT-Settl grid (M dwarf, optional comparison) | `<WORKSPACE>/model_cache/` | ~2.2 GB downloaded, ~190 MB on disk | only if the settings use `btsettl`, or if you force it below |
| SVO filter curves of the SED photometry | `<WORKSPACE>/model_cache/filters/` | < 1 MB | automatic during the first run (they depend on the target's photometry) |

Downloads can be interrupted and restarted: files already present are kept. Cell 1 holds every choice; the others can be run as they are.

## 1. Settings

In [ ]:
# Settings: change these before running the cells below.
DOWNLOADS_DIR            = None   # None = your Downloads folder; or e.g. "/path/to/folder"
INSTALL_MISSING_PACKAGES = False  # True: install missing Python packages with pip (cell 3)

GET_KOESTER     = True   # Koester DA spectra: required by the pipeline
GET_PHOENIX     = None   # None = only if main_all.py uses PHOENIX (default settings: yes)
GET_BTSETTL     = None   # None = only if main_all.py uses BT-Settl (default settings: no)
BUILD_SED_GRIDS = True   # also prepare the rebinned grids of the SED fit (saves time later)

N_PARALLEL_DOWNLOADS = 4      # simultaneous downloads from SVO
MAX_KOESTER_FILES    = None   # None = all 1 066 files; a small number for a quick trial

## 2. Locate the repository and the workspace

The pipeline's own rules are used (`2_sed_dilution/wdms_paths.py`), and its settings are read from `main_all.py` (reading it computes nothing).

In [ ]:
import os
import sys
import shutil
import importlib.util
from pathlib import Path


def find_code_dir():
    """<repository>/2_sed_dilution/, searched from the notebook's folder upwards."""
    env = os.environ.get("WDMS_REPO_DIR")
    roots = ([Path(env)] if env else []) + [Path.cwd(), *Path.cwd().parents]
    for folder in roots:
        for candidate in (folder, folder / "2_sed_dilution"):
            if (candidate / "wdms_paths.py").is_file():
                return candidate.resolve()
    raise FileNotFoundError("2_sed_dilution/ not found: open this notebook from the repository "
                            "(0_setup/), or set the environment variable WDMS_REPO_DIR.")


CODE_DIR = find_code_dir()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
import wdms_paths as paths
import main_all as cfg            # pipeline settings only

DOWNLOADS = paths.resolve_downloads_dir(DOWNLOADS_DIR, "DOWNLOADS_DIR")
WORKSPACE = paths.ensure_dir(paths.workspace_dir(DOWNLOADS), "workspace")
MODELS    = paths.ensure_dir(Path(cfg.MODELS_DIR) if cfg.MODELS_DIR else paths.models_dir(DOWNLOADS),
                             "models folder")
CACHE     = paths.ensure_dir(Path(cfg.SED_ATMO_CACHE_DIR) if cfg.SED_ATMO_CACHE_DIR
                             else paths.atmosphere_cache_dir(DOWNLOADS), "model cache")
KOESTER   = MODELS / "koester2"

print(f"Code folder : {CODE_DIR}")
print(f"Workspace   : {WORKSPACE}")
print(f"Models      : {MODELS}")
print(f"Model cache : {CACHE}")
print(f"Free disk space: {shutil.disk_usage(WORKSPACE).free / 1e9:.1f} GB")

## 3. Python packages

Lists what is installed and what is missing. With `INSTALL_MISSING_PACKAGES = True`, the missing ones are installed with `pip` into the Python running this notebook. Versions are not pinned; the code needs NumPy ≥ 2.0.

In [ ]:
import subprocess

REQUIRED = {  # import name: pip name
    "numpy": "numpy", "scipy": "scipy", "matplotlib": "matplotlib", "pandas": "pandas",
    "astropy": "astropy", "astroquery": "astroquery", "ultranest": "ultranest",
    "certifi": "certifi", "requests": "requests",
}
OPTIONAL = {  # composite figure, and the Part 1 notebook
    "PIL": "pillow", "fitz": "pymupdf", "lightkurve": "lightkurve", "wotan": "wotan",
}

def missing(packages):
    return {imp: pip for imp, pip in packages.items() if importlib.util.find_spec(imp) is None}

for title, packages in (("Required", REQUIRED), ("Optional", OPTIONAL)):
    absent = missing(packages)
    print(f"{title:9s}: {len(packages) - len(absent)}/{len(packages)} installed"
          + (f" — missing: {', '.join(absent.values())}" if absent else ""))

to_install = list({**missing(REQUIRED), **missing(OPTIONAL)}.values())
if to_install and INSTALL_MISSING_PACKAGES:
    print("Installing:", " ".join(to_install))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *to_install])
    importlib.invalidate_caches()
elif to_install:
    print(f"To install them: set INSTALL_MISSING_PACKAGES = True, or run\n"
          f"    {sys.executable} -m pip install {' '.join(to_install)}")

if importlib.util.find_spec("numpy") is not None:
    import numpy
    if int(numpy.__version__.split(".")[0]) < 2:
        print(f"WARNING: NumPy {numpy.__version__} is installed; the pipeline needs NumPy >= 2.0.")

## 4. What will be downloaded

Decides which grids are needed from the pipeline settings and from cell 1, then estimates the sizes and compares them with the free disk space. Nothing is downloaded yet.

In [ ]:
import io
import re
import numpy as np
import requests
from astropy.io.votable import parse

SVO_SSAP = "http://svo2.cab.inta-csic.es/theory/newov2/ssap.php"
MB = 1e6

def svo_index(model):
    """(teff, logg, fid) of every spectrum of an SVO theoretical model."""
    r = requests.get(f"{SVO_SSAP}?model={model}", timeout=180)
    r.raise_for_status()
    t = parse(io.BytesIO(r.content)).get_first_table().to_table()
    fids = [int(re.search(r"fid=(\d+)", str(ref)).group(1)) for ref in t["Access.Reference"]]
    return [(float(a), float(b), f) for a, b, f in zip(t["teff"], t["logg"], fids)]

def koester_path(teff, logg):
    return KOESTER / f"da{int(round(teff)):05d}_{int(round(100 * logg)):03d}.dk.dat.txt"

# Companion models used by the pipeline settings (main_all.py)
variants = {str(cfg.SED_MS_ATMOSPHERE).lower()}
compare = cfg.SED_COMPARE_ATMOSPHERES
if isinstance(compare, str):
    variants |= {"phoenix", "btsettl"} if compare.lower() == "all" else {compare.lower()}
elif compare:
    variants |= {str(v).lower() for v in compare}
uses_phoenix = bool(variants & {"phoenix", "phoenix-stsci"})
uses_btsettl = bool(variants & {"btsettl", "bt-settl", "bt-settl-cifist"})
want_phoenix = uses_phoenix if GET_PHOENIX is None else bool(GET_PHOENIX)
want_btsettl = uses_btsettl if GET_BTSETTL is None else bool(GET_BTSETTL)

import sed_atmosphere_models as sam
from sed_fit import ATMO_GRID_RANGES, WD_ATMO_GRID_RANGE

def grid_range(model, default):
    r = dict(default)
    r.update((cfg.SED_ATMO_GRID_RANGES or {}).get(model, {}))
    return r

plan = []

# Koester DA (SVO): ~0.2 MB per VOTable download, ~0.13 MB per text file on disk
koester_index = svo_index("koester2") if GET_KOESTER else []
if MAX_KOESTER_FILES:
    koester_index = koester_index[:int(MAX_KOESTER_FILES)]
koester_todo = [e for e in koester_index if not (koester_path(e[0], e[1]).is_file()
                                                 and koester_path(e[0], e[1]).stat().st_size > 0)]
plan.append(("Koester DA (required)", GET_KOESTER, len(koester_index), len(koester_todo),
             0.21 * MB * len(koester_todo), 0.135 * MB * len(koester_todo)))

# STScI PHOENIX: one FITS file per Teff; exact sizes from the server
phoenix_range = grid_range("phoenix-stsci", ATMO_GRID_RANGES["phoenix-stsci"])
phoenix_src = sam._PhoenixStScI()
phoenix_teff, _ = phoenix_src.nodes(phoenix_range["teff"], phoenix_range["logg"])
phoenix_todo = [t for t in phoenix_teff if not phoenix_src.raw_path(t, CACHE).is_file()]
phoenix_bytes = 0.0
if want_phoenix:
    for t in phoenix_todo:
        h = requests.head(f"{phoenix_src.base_url}/phoenixm00_{int(round(t))}.fits", timeout=60)
        phoenix_bytes += float(h.headers.get("Content-Length", 11.5 * MB))
plan.append((f"STScI PHOENIX ({'used' if uses_phoenix else 'not used'} by the settings)", want_phoenix,
             len(phoenix_teff), len(phoenix_todo), phoenix_bytes, phoenix_bytes))

# BT-Settl CIFIST (SVO): ~66 MB per (Teff, logg) download, ~5.5 MB kept per node
btsettl_range = grid_range("bt-settl-cifist", ATMO_GRID_RANGES["bt-settl-cifist"])
btsettl_nodes = []
if want_btsettl:
    btsettl_nodes = [e for e in svo_index("bt-settl-cifist")
                     if btsettl_range["teff"][0] <= e[0] <= btsettl_range["teff"][1]
                     and btsettl_range["logg"][0] <= e[1] <= btsettl_range["logg"][1]]
btsettl_todo = [e for e in btsettl_nodes
                if not sam._BtSettlCifistSVO().raw_path(e[0], e[1], CACHE).is_file()]
plan.append((f"BT-Settl ({'used' if uses_btsettl else 'not used'} by the settings)", want_btsettl,
             len(btsettl_nodes), len(btsettl_todo), 66 * MB * len(btsettl_todo), 5.5 * MB * len(btsettl_todo)))

print(f"{'Component':42s} {'selected':>8s} {'files':>6s} {'to get':>6s} {'download':>10s} {'disk':>9s}")
total_dl = total_disk = 0.0
for name, selected, n_all, n_todo, dl, disk in plan:
    if selected:
        total_dl += dl; total_disk += disk
    print(f"{name:42s} {'yes' if selected else 'no':>8s} {n_all:6d} {n_todo:6d} "
          f"{dl / MB:8.0f} MB {disk / MB:7.0f} MB")
free = shutil.disk_usage(WORKSPACE).free
print(f"\nTotal for the selected components: ~{total_dl / MB:.0f} MB to download, "
      f"~{total_disk / MB:.0f} MB on disk; free space {free / 1e9:.1f} GB")
if total_disk > 0.9 * free:
    print("WARNING: not enough free disk space for this selection.")

## 5. Koester DA spectra (SVO)

Each spectrum is downloaded from the SVO server and saved as `da<Teff>_<100·logg>.dk.dat.txt` in the text format the pipeline reads (wavelength in Å, surface flux in erg s⁻¹ cm⁻² Å⁻¹). These files were checked against an existing local copy: the fluxes agree to better than 10⁻⁷.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

HEADER = ("# Koester WD models\n"
          "# teff = {teff:.0f} K (value for the effective temperature for the model. "
          "Temperatures are given in K)\n"
          "# logg = {logg:g} log(cm/s2) (value for Log(G) for the model.)\n"
          "#\n"
          "# column 1: WAVELENGTH (ANGSTROM), Wavelength in Angstrom\n"
          "# column 2: FLUX (ERG/CM2/S/A), Flux in erg/cm2/s/A\n")

def fetch_koester(teff, logg, fid, retries=3):
    path = koester_path(teff, logg)
    if path.is_file() and path.stat().st_size > 0:
        return "kept"
    path.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(retries):
        try:
            r = requests.get(f"{SVO_SSAP}?model=koester2&fid={fid}", timeout=180)
            r.raise_for_status()
            t = parse(io.BytesIO(r.content)).get_first_table().to_table()
            wave = np.asarray(t["WAVELENGTH"], float)
            flux = np.asarray(t["FLUX"], float)
            order = np.argsort(wave)
            tmp = path.with_suffix(".part")
            with open(tmp, "w", encoding="utf-8") as fh:
                fh.write(HEADER.format(teff=teff, logg=logg))
                for w, f in zip(wave[order], flux[order]):
                    fh.write(f"{w:15.3f} {f:15.5e}\n")
            tmp.replace(path)                     # complete files only
            return "downloaded"
        except Exception as exc:                  # network hiccup: retry
            if attempt + 1 == retries:
                return f"FAILED ({exc})"
            time.sleep(2.0 * (attempt + 1))

if not GET_KOESTER:
    print("Koester download skipped (GET_KOESTER = False).")
else:
    t0 = time.time(); counts = {}
    with ThreadPoolExecutor(max_workers=int(N_PARALLEL_DOWNLOADS)) as pool:
        futures = {pool.submit(fetch_koester, *e): e for e in koester_index}
        for k, fut in enumerate(as_completed(futures), start=1):
            status = fut.result().split(" ")[0]
            counts[status] = counts.get(status, 0) + 1
            if status == "FAILED":
                print("  failed:", futures[fut], fut.result())
            if k % 100 == 0 or k == len(futures):
                print(f"  {k}/{len(futures)} spectra  {counts}  ({time.time() - t0:.0f} s)")
    n_files = len(list(KOESTER.glob("da*.dk.dat.txt")))
    print(f"Koester folder: {KOESTER} — {n_files} spectra"
          + ("" if counts.get("FAILED") is None else " — rerun this cell to retry the failures"))

## 6. M-dwarf grids (PHOENIX, BT-Settl) and SED grids

Uses the pipeline's own grid builder (`sed_atmosphere_models.SpectralGrid.load`), with the same ranges and cache folder as `main_all.py`. The pipeline then finds these grids already built. Skipped entries are downloaded automatically at the first run if the settings need them.

In [ ]:
import time

os.environ[paths.MODELS_ENV_VAR] = str(MODELS)   # where the SED grid builder reads Koester

def build(name, key, rng, selected):
    if not selected:
        print(f"{name}: skipped")
        return
    t0 = time.time()
    grid = sam.SpectralGrid.load(source_key=key, teff_range=rng["teff"], logg_range=rng["logg"],
                                 cache_dir=CACHE, verbose=False)
    print(f"{name}: ready — Teff {grid.teff_bounds[0]:.0f}–{grid.teff_bounds[1]:.0f} K, "
          f"logg {grid.logg_bounds[0]:.1f}–{grid.logg_bounds[1]:.1f} ({time.time() - t0:.0f} s)")

build("STScI PHOENIX", "phoenix-stsci", phoenix_range, want_phoenix)
build("BT-Settl CIFIST", "bt-settl-cifist", btsettl_range, want_btsettl)
koester_ok = len(list(KOESTER.glob("da*.dk.dat.txt"))) >= 1066
build("Koester DA grid for the SED fit", "koester-da",
      grid_range("koester-da", WD_ATMO_GRID_RANGE), BUILD_SED_GRIDS and koester_ok)
if BUILD_SED_GRIDS and not koester_ok:
    print("  (needs the complete Koester folder: run cell 5 first)")

## 7. Final check

Runs the pipeline's input check on the example target (`main_all.py --example --check`): it lists anything still missing and computes nothing. When it passes, continue with Part 1 (`1_lightcurves_environment/Light_curves_analysis.ipynb`) and Part 2 (`2_sed_dilution/main_all.py`).

In [ ]:
import subprocess

cmd = [sys.executable, str(CODE_DIR / "main_all.py"), "--example", "--check"]
if DOWNLOADS_DIR:
    cmd += ["--downloads-dir", str(DOWNLOADS_DIR)]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-4000:] or result.stderr[-4000:])